In [ ]:
# @title 1. Setup and Configuration
# --- IMPORTANT ---
# Change the value of `zip_file_path` to the actual path of your uploaded zip file.

# If you are using Google Colab, you can upload your file to the session storage
# by clicking the "Files" icon on the left sidebar and then "Upload to session storage".
# The path will typically be "/content/your_file_name.zip".

zip_file_path = "/content/images.zip"  # <-- CHANGE THIS TO YOUR FILE'S PATH

# --- Library Imports ---
import os
import zipfile
from PIL import Image
import io
from IPython.display import display, FileLink

print("Configuration and libraries are ready.")
print(f"Will attempt to process file at: {zip_file_path}")

In [ ]:
# @title 2. Process Images and Create New Zip Files
# This is the main part of our notebook.
# It will:
# - Read the zip file from the path specified above.
# - Crop the images in two different ways.
# - Create two new zip files with the results.
# - Provide download links.

def process_images_from_path(path_to_zip):
    # First, check if the file actually exists at the given path
    if not os.path.exists(path_to_zip):
        print(f"🚨 ERROR: File not found at '{path_to_zip}'")
        print("Please make sure the path in the first cell is correct and you have uploaded the file.")
        return

    print(f"Processing images from: {path_to_zip}")

    # Define the names for our output zip files
    right_half_zip_name = 'cropped_right_half.zip'
    top_right_quadrant_zip_name = 'cropped_top_right_quadrant.zip'

    # Create in-memory zip files to store the cropped images
    right_half_zip_buffer = io.BytesIO()
    top_right_quadrant_zip_buffer = io.BytesIO()

    # Use a 'try...except' block to handle potential errors with the zip file itself
    try:
        with zipfile.ZipFile(path_to_zip, 'r') as original_zip:
            with zipfile.ZipFile(right_half_zip_buffer, 'w') as right_half_zip, \
                 zipfile.ZipFile(top_right_quadrant_zip_buffer, 'w') as top_right_quadrant_zip:

                image_files_found = 0
                for item in original_zip.infolist():
                    # Check if the file is a common image type and not a directory
                    if not item.is_dir() and item.filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif', '.tiff')):
                        image_files_found += 1
                        try:
                            with original_zip.open(item) as image_file:
                                img = Image.open(image_file).convert("RGB") # Convert to RGB to handle formats like RGBA or P
                                width, height = img.size

                                # --- Crop Right Half ---
                                # Coords: (left, upper, right, lower)
                                right_half_coords = (width / 2, 0, width, height)
                                cropped_right = img.crop(right_half_coords)
                                # Save to a byte buffer
                                right_buffer = io.BytesIO()
                                cropped_right.save(right_buffer, format='JPEG') # Save as JPEG for consistency
                                right_half_zip.writestr(f"right_half_{os.path.basename(item.filename)}", right_buffer.getvalue())

                                # --- Crop Top Right Quadrant ---
                                top_right_coords = (width / 2, 0, width, height / 2)
                                cropped_top_right = img.crop(top_right_coords)
                                # Save to a byte buffer
                                top_right_buffer = io.BytesIO()
                                cropped_top_right.save(top_right_buffer, format='JPEG')
                                top_right_quadrant_zip.writestr(f"top_right_{os.path.basename(item.filename)}", top_right_buffer.getvalue())

                        except Exception as e:
                            print(f"Could not process file {item.filename}: {e}")

        if image_files_found == 0:
            print("⚠️ Warning: No image files (.png, .jpg, etc.) were found in the zip archive.")
            return

        # Save the in-memory zip files to disk
        with open(right_half_zip_name, 'wb') as f:
            f.write(right_half_zip_buffer.getvalue())
        with open(top_right_quadrant_zip_name, 'wb') as f:
            f.write(top_right_quadrant_zip_buffer.getvalue())

        print("\n✅ Processing complete!")
        print(f"Created '{right_half_zip_name}' and '{top_right_quadrant_zip_name}'.")

        # --- Display Download Links ---
        print("\n⬇️ Download your new zip files:")
        display(FileLink(right_half_zip_name))
        display(FileLink(top_right_quadrant_zip_name))

    except zipfile.BadZipFile:
        print(f"🚨 ERROR: The file '{path_to_zip}' is not a valid zip file.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

# --- Run the main function ---
process_images_from_path(zip_file_path)